In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# INTERNSHIP SEARCH ENGINE - UK
# A separate, internship-only build of the main job-search notebook. It reuses
# that notebook's Adzuna/Reed clients, LinkedIn reader, scoring shape,
# deduplication and Excel/hyperlink writer, and adds:
#   - internship-specific search vocabulary and exclusions
#   - an eligibility reader (undergraduate-only vs postgraduate/graduate open)
#   - deadline parsing and urgency status
#   - student/graduate job boards
# Run the cells top to bottom in Google Colab. Only the export cell downloads.
# ─────────────────────────────────────────────────────────────────────────────
!pip install requests pandas openpyxl beautifulsoup4 lxml -q
print("Packages ready.")


In [ ]:
# ── CANDIDATE PROFILE (from the CV only) ─────────────────────────────────────
# Riya Singh - UK based.
#   MSc Programme and Project Management, University of Warwick, Sept 2025 - Sept 2026
#     -> a CURRENTLY ENROLLED postgraduate student for the whole of this search.
#   BTech Information Technology, SRM Institute of Science and Technology, 2017-2021.
#   Wells Fargo, Data Management Analyst, Feb 2024 - Jul 2025:
#     FR Y-14 Schedule H1/H2 submissions, FR Y-14Q and BCBS 239 data quality
#     checks, regulatory data reconciliation, month-end reporting, governance
#     documentation, audit trails, 45 audit-ready reports, complex SQL.
#   Mu Sigma, Business Analyst, Jul 2021 - Feb 2024:
#     data governance framework build, metadata repositories, information
#     standards, executive summaries, stakeholder management on ML delivery.
#   TCS intern, Dec 2018 - Jan 2019: financial/regulatory data frameworks exposure.
#   Projects: GTM strategy lead (Warwick x Retrogreen, Singapore market entry);
#     co-founder of an AI beauty e-commerce platform, national start-up winner.
#   Tools: SQL, Python, Advanced Excel, VBA, Power BI, Tableau, Alteryx,
#     SharePoint, ServiceNow, JIRA, Trello.
#   Certifications: APM, Machine Learning, Data Analytics with Power BI.
# NOT on the CV, so never assumed: visa or right-to-work status, salary
# expectations, people-management scope, accountancy or actuarial qualifications,
# software engineering experience.
# ─────────────────────────────────────────────────────────────────────────────

import os

# Colab Secrets / environment variables take precedence over anything stored here.
def _secret(name, default=""):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

# ── CREDENTIALS ──────────────────────────────────────────────────────────────
# Paste between the quotes, or add the same names under Colab Secrets.
# Only Adzuna and Reed need keys. Every other source here is keyless.
ADZUNA_APP_ID  = _secret("ADZUNA_APP_ID",  "")   # <-- paste your Adzuna app id here
ADZUNA_APP_KEY = _secret("ADZUNA_APP_KEY", "")   # <-- paste your Adzuna app key here
REED_API_KEY   = _secret("REED_API_KEY",   "")   # <-- paste your Reed API key here

# ── SEARCH SETTINGS ──────────────────────────────────────────────────────────
# London first, then the rest of the UK: a strong internship in Manchester or
# Edinburgh should still surface.
SEARCH_LOCATIONS   = ["London", "United Kingdom"]
RESULTS_PER_SOURCE = 25
SALARY_MIN         = 0          # internships are often unpaid or bursary-based
OUTPUT_FILE        = "riya_internships.xlsx"
AI_OUTPUT_FILE     = "riya_internships_ai_review.xlsx"

# Deadlines within this many days count as "Closing Soon".
CLOSING_SOON_DAYS = 14
# Match score a role needs before it can reach the Top Matches sheet.
TOP_MATCH_MIN_SCORE = 55

# ── INTERNSHIP ROLE CATEGORIES ───────────────────────────────────────────────
# Each category carries the title vocabulary used for searching AND for scoring.
ROLE_CATEGORIES = {
    "Data / Analytics": [
        "data analyst", "data analytics", "analytics", "data intern", "data science",
        "business intelligence", "bi analyst", "power bi", "tableau", "insight",
        "data governance", "data quality", "data management", "data steward",
        "reporting analyst", "management information", "mi analyst",
    ],
    "Risk / Regulatory / Financial Data": [
        "risk analytics", "risk data", "risk intern", "regulatory reporting",
        "regulatory data", "financial data", "financial analyst", "finance intern",
        "banking operations", "financial services", "credit risk", "compliance data",
        "financial crime", "controls",
    ],
    "Business Analysis": [
        "business analyst", "business analysis", "business data analyst",
        "business process", "process analyst", "requirements",
    ],
    "Project / PMO": [
        "project analyst", "project management", "pmo", "programme management",
        "program management", "project intern", "portfolio", "project support",
        "transformation project",
    ],
    "Strategy / Transformation / Consulting": [
        "strategy", "strategy and operations", "strategy & operations",
        "business operations", "transformation", "change", "commercial strategy",
        "consulting", "consultant", "advisory", "operations intern",
    ],
}

CATEGORY_TITLE_TERMS = sorted({term for terms in ROLE_CATEGORIES.values() for term in terms})

# Words that make a posting an internship. Checked against the TITLE first, so a
# permanent role that merely mentions "our internship programme" is not pulled in.
INTERNSHIP_TITLE_SIGNALS = [
    "intern", "internship", "placement", "summer analyst", "summer associate",
    "industrial placement", "work placement", "student placement", "sandwich",
    "vacation scheme", "insight programme", "insight program", "spring week",
    "summer programme", "summer program", "trainee", "graduate intern",
]

# Search strings sent to the job boards. Deliberately high recall; precision is
# applied afterwards by score_internship() and the exclusion layer.
INTERNSHIP_SEARCH_KEYWORDS = [
    # Data / analytics
    "data analyst intern", "data analytics intern", "data intern",
    "business intelligence intern", "bi intern", "analytics intern",
    "data governance intern", "data quality intern", "data management intern",
    "data analyst placement", "data analytics placement",
    # Risk / regulatory / financial data
    "risk analytics intern", "risk intern", "risk data intern",
    "regulatory reporting intern", "regulatory data intern",
    "financial data intern", "financial services intern",
    "banking operations intern", "finance intern",
    # Business analysis
    "business analyst intern", "business analysis placement",
    "business analyst placement",
    # Project / PMO
    "project analyst intern", "project management intern", "pmo intern",
    "programme management intern", "transformation intern",
    # Strategy / consulting
    "strategy intern", "strategy and operations intern",
    "business operations intern", "commercial strategy intern",
    "consulting intern", "summer analyst", "summer associate",
]

# ── EXCLUSIONS ───────────────────────────────────────────────────────────────
# HARD: never right for this CV.
HARD_EXCLUDE = [
    "software engineer", "software developer", "software engineering",
    "frontend", "front end developer", "backend", "back end developer",
    "full stack", "fullstack", "mobile developer", "ios developer",
    "android developer", "game developer", "web developer", "devops",
    "site reliability", "security engineer", "network engineer", "qa engineer",
    "test engineer", "hardware", "embedded", "firmware",
    "mechanical engineer", "civil engineer", "electrical engineer",
    "chemical engineer", "aerospace", "structural engineer", "manufacturing engineer",
    "laboratory", "lab technician", "chemist", "biologist", "biomedical",
    "clinical", "nursing", "nurse", "medical", "pharmacy", "dental",
    "veterinary", "physiotherapy", "psychology assistant",
    "teaching", "teacher", "teaching assistant", "tutor", "lecturer",
    "sales executive", "sales intern", "telesales", "field sales",
    "business development representative", "sdr ", "recruitment consultant",
    "social media intern", "content creator", "copywriter", "graphic design",
    "fashion", "journalism", "photography", "video editor",
    "architecture intern", "quantity surveying", "construction",
    "chef", "hospitality", "retail assistant", "warehouse", "driver",
]

# SOFT: dropped only when nothing ties the advert back to this CV.
SOFT_EXCLUDE = [
    "marketing", "brand", "pr intern", "human resources", "hr intern",
    "recruitment", "audit intern", "tax intern", "accounting intern",
    "legal intern", "law intern", "supply chain", "procurement",
    "machine learning", "data scientist", "ai research",
]

# Any of these anywhere in title + description rescues a SOFT-excluded advert.
RESCUE_TERMS = [
    "data analyst", "data analytics", "data governance", "data quality",
    "data management", "business analyst", "business analysis", "sql",
    "power bi", "tableau", "regulatory reporting", "regulatory data",
    "risk data", "reporting", "management information", "reconciliation",
    "governance", "stakeholder", "pmo", "project analyst", "financial services",
    "banking", "excel", "dashboard", "alteryx",
]

# Technical depth the CV does not evidence. Lowers the score, never removes a row.
UNSUPPORTED_TECH_TERMS = [
    "spark", "scala", "hadoop", "kafka", "airflow", "kubernetes", "terraform",
    "microservices", "deep learning", "pytorch", "tensorflow", "computer vision",
    "nlp research", "c++", "golang", "rust ", "java developer", "react", "node.js",
]

# ── SKILLS THE CV EVIDENCES (used for "Key Matching Skills") ─────────────────
CV_SKILLS = {
    "SQL": ["sql", "queries", "database", "query"],
    "Python": ["python", "pandas"],
    "Advanced Excel / VBA": ["excel", "vba", "macros", "pivot", "spreadsheet"],
    "Power BI": ["power bi", "powerbi"],
    "Tableau": ["tableau"],
    "Alteryx": ["alteryx"],
    "Data Governance": ["data governance", "governance framework", "data policy", "data steward"],
    "Data Quality": ["data quality", "data validation", "data integrity", "data cleansing"],
    "Data Management / MDM": ["data management", "master data", "metadata", "reference data", "data lineage"],
    "Regulatory Reporting": ["regulatory reporting", "regulatory return", "fr y-14", "bcbs", "ccar", "corep", "finrep"],
    "Reconciliation & Controls": ["reconciliation", "reconcile", "month-end", "controls", "audit trail"],
    "Business Analysis": ["business analysis", "business analyst", "requirements", "process mapping", "uat"],
    "Project / PMO": ["project", "programme", "pmo", "milestone", "jira", "agile", "waterfall", "prince2"],
    "Stakeholder Management": ["stakeholder", "cross-functional", "senior", "presentation", "communication"],
    "Financial Services Domain": ["bank", "banking", "financial services", "capital markets", "insurance", "fintech"],
}

# Requirements an advert may ask for that the CV does not evidence.
GAP_REQUIREMENTS = {
    "Software engineering / production coding": ["software development", "production code", "ci/cd", "microservices", "api development"],
    "Advanced ML / modelling": ["machine learning model", "deep learning", "statistical modelling", "predictive model", "data scientist"],
    "Cloud / big data engineering": ["aws", "azure", "gcp", "spark", "databricks", "snowflake", "hadoop"],
    "Accounting qualification": ["aca", "acca", "cima", "chartered accountant", "qualified accountant"],
    "Actuarial / CFA": ["actuarial", "cfa"],
    "Undergraduate study in progress": ["penultimate year", "second year undergraduate", "first year undergraduate"],
    "Specific degree discipline": ["stem degree", "engineering degree", "computer science degree", "mathematics degree"],
}

print("Internship profile loaded for Riya Singh (MSc student to Sept 2026).")
print(f"  Role categories:     {len(ROLE_CATEGORIES)}")
print(f"  Search keywords:     {len(INTERNSHIP_SEARCH_KEYWORDS)}")
print(f"  Locations:           {', '.join(SEARCH_LOCATIONS)}")
print(f"  Hard exclusions:     {len(HARD_EXCLUDE)} | Soft (rescuable): {len(SOFT_EXCLUDE)}")


In [ ]:
# ── SHARED HELPERS ───────────────────────────────────────────────────────────
# Carried over from the main job-search notebook so this build stands alone.
import re
import json
import time
import requests
from bs4 import BeautifulSoup

HEADERS_BROWSER = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-GB,en;q=0.9",
}

# Every source writes its outcome here, and the export cell prints the summary.
SOURCE_STATUS = {}


def note_source(name, count, detail=""):
    SOURCE_STATUS[name] = {"roles": count, "detail": detail}
    flag = "ok  " if count else "none"
    print(f"  [{flag}] {name:<28} {count:>4} roles  {detail}")


def normalise_text(value):
    return re.sub(r"\s+", " ", str(value or "").lower()).strip()


def clean_description(raw):
    """HTML to readable text. Nothing is trimmed - length handling is Excel's problem."""
    if not raw:
        return ""
    text = BeautifulSoup(str(raw), "html.parser").get_text("\n", strip=True)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def request_with_backoff(url, retries=3, base_sleep=1.0, method="get", **kwargs):
    """Retries the retryable status codes, then gives up rather than raising upward."""
    for attempt in range(retries):
        try:
            caller = requests.post if method == "post" else requests.get
            response = caller(url, **kwargs)
            if response.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"Retryable status {response.status_code}")
            return response
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(base_sleep * (2 ** attempt))


# ── UK LOCATION FILTERING ────────────────────────────────────────────────────
UK_LOCATION_SIGNALS = [
    "london", "united kingdom", "uk", "england", "scotland", "wales",
    "northern ireland", "remote uk", "remote - uk", "hybrid", "manchester",
    "edinburgh", "birmingham", "bristol", "leeds", "sheffield", "cardiff",
    "glasgow", "coventry", "oxford", "cambridge", "belfast", "newcastle",
    "nottingham", "reading", "milton keynes", "liverpool", "southampton",
    "brighton", "york", "aberdeen", "swansea", "warwick", "watford", "slough",
]

NON_UK_SIGNALS = [
    "united states", " usa", "u.s.", " us,", "canada", "australia", "japan",
    "singapore", "india", "germany", "france", "spain", "italy", "mexico",
    "brazil", "new york", "san francisco", "seattle", "boston", "chicago",
    "los angeles", "austin", "tokyo", "sydney", "melbourne", "toronto", "paris",
    "berlin", "barcelona", "rome", "gurugram", "gurgaon", "mumbai", "bangalore",
    "bengaluru", "hyderabad", "delhi", "munich", "frankfurt", "dublin", "ireland",
    "lisbon", "stockholm", "amsterdam", "netherlands", "belgium", "poland",
    "warsaw", "madrid", "milan", "zurich", "switzerland", "shanghai", "beijing",
    "hong kong", "seoul", "dubai", "abu dhabi", "riyadh", "remote - us", "remote us",
]


def is_uk_location(location):
    """Unknown locations are kept and reviewed later rather than silently dropped."""
    loc = normalise_text(location)
    if not loc:
        return True
    if any(signal in loc for signal in NON_UK_SIGNALS):
        return False
    return True


def location_priority(location):
    """1 London, 2 UK-wide/other UK, 3 remote/hybrid UK, 4 unknown."""
    loc = normalise_text(location)
    if "london" in loc:
        return 1, "London"
    if any(signal in loc for signal in ["remote", "hybrid", "work from home"]):
        return 3, "Remote / Hybrid UK"
    if any(signal in loc for signal in UK_LOCATION_SIGNALS):
        return 2, "UK (outside London)"
    if not loc:
        return 4, "Unknown"
    return 4, "Unknown"


# ── EXCLUSIONS ───────────────────────────────────────────────────────────────
def term_hits(text, terms):
    return [term for term in terms if term in text]


def exclusion_reason(title, description=""):
    """
    Hard terms always exclude. Soft terms exclude only when nothing in the advert
    ties it back to this CV, so a "Marketing Data Analyst Intern" survives while a
    "Marketing Intern" does not.
    """
    t = normalise_text(title)
    combined = f"{t} {normalise_text(description)}"
    hard = term_hits(t, HARD_EXCLUDE)
    if hard:
        return f"Excluded - {hard[0]}"
    soft = term_hits(t, SOFT_EXCLUDE)
    if soft and not term_hits(combined, RESCUE_TERMS):
        return f"Excluded - {soft[0]} with no data/analysis content"
    return ""


def is_internship(title, description="", employment_type=""):
    """
    The internship signal must come from the title or from the structured
    employment type. A permanent advert that merely mentions "our internship
    programme" in its body is not an internship and does not belong here.
    """
    t = normalise_text(title)
    if any(signal in t for signal in INTERNSHIP_TITLE_SIGNALS):
        return True
    et = normalise_text(employment_type)
    return any(signal in et for signal in ["intern", "placement", "student"])


# ── JSON-LD JobPosting ───────────────────────────────────────────────────────
# Most UK job boards embed schema.org JobPosting markup. Reading that is more
# stable than CSS selectors and hands us the description, the closing date and
# the employment type in one go.
def _iter_json_ld(soup):
    for tag in soup.find_all("script", attrs={"type": "application/ld+json"}):
        raw = tag.string or tag.get_text() or ""
        if not raw.strip():
            continue
        try:
            data = json.loads(raw)
        except Exception:
            continue
        stack = [data]
        while stack:
            node = stack.pop()
            if isinstance(node, list):
                stack.extend(node)
            elif isinstance(node, dict):
                if "@graph" in node:
                    stack.extend(node["@graph"] if isinstance(node["@graph"], list) else [node["@graph"]])
                yield node


def _first_text(value):
    if isinstance(value, dict):
        for key in ("name", "@id", "url", "value"):
            if key in value:
                return _first_text(value[key])
        return ""
    if isinstance(value, list):
        return ", ".join(filter(None, (_first_text(v) for v in value)))
    return str(value or "")


def _location_from_jsonld(node):
    location = node.get("jobLocation")
    parts = []
    for entry in (location if isinstance(location, list) else [location]):
        if not isinstance(entry, dict):
            parts.append(_first_text(entry))
            continue
        address = entry.get("address", {})
        if isinstance(address, dict):
            parts.append(", ".join(filter(None, [
                str(address.get("addressLocality", "") or ""),
                str(address.get("addressRegion", "") or ""),
                str(address.get("addressCountry", "") or "")
                if isinstance(address.get("addressCountry"), str)
                else _first_text(address.get("addressCountry", "")),
            ])).strip(", "))
        else:
            parts.append(_first_text(address))
    if str(node.get("jobLocationType", "")).upper().find("TELECOMMUTE") >= 0:
        parts.append("Remote")
    return "; ".join(p for p in parts if p).strip("; ") or ""


def _salary_from_jsonld(node):
    salary = node.get("baseSalary")
    if not isinstance(salary, dict):
        return _first_text(salary)
    value = salary.get("value", {})
    if isinstance(value, dict):
        low = value.get("minValue", "")
        high = value.get("maxValue", "")
        single = value.get("value", "")
        unit = value.get("unitText", "")
        if low or high:
            return f"{low}-{high} {unit}".strip()
        if single:
            return f"{single} {unit}".strip()
    return _first_text(salary.get("currency", ""))


def parse_jsonld_jobs(html, source, page_url=""):
    """Returns every schema.org JobPosting embedded in a page."""
    jobs = []
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception:
        return jobs
    for node in _iter_json_ld(soup):
        types = node.get("@type", "")
        types = types if isinstance(types, list) else [types]
        if not any(str(t).lower() == "jobposting" for t in types):
            continue
        jobs.append({
            "title": _first_text(node.get("title", "")),
            "company": _first_text(node.get("hiringOrganization", "")),
            "location": _location_from_jsonld(node),
            "salary": _salary_from_jsonld(node),
            "posted": str(node.get("datePosted", ""))[:10],
            "deadline_raw": str(node.get("validThrough", ""))[:10],
            "employment_type": _first_text(node.get("employmentType", "")),
            "desc": clean_description(node.get("description", "")),
            "url": _first_text(node.get("url", "")) or page_url,
            "source": source,
        })
    return jobs


def harvest_job_links(html, base_url, source, link_hints=("/job", "/jobs/", "/vacancy", "/placement", "/internship")):
    """
    Fallback for pages with no JSON-LD: collect anchors that look like job links.
    Title-only rows, flagged as such, rather than nothing at all.
    """
    jobs = []
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception:
        return jobs
    from urllib.parse import urljoin
    seen = set()
    for anchor in soup.find_all("a", href=True):
        title = anchor.get_text(" ", strip=True)
        href = anchor["href"]
        if len(title) < 8 or len(title) > 120:
            continue
        if not any(hint in href.lower() for hint in link_hints):
            continue
        full_url = urljoin(base_url, href)
        if full_url in seen:
            continue
        seen.add(full_url)
        jobs.append({
            "title": title, "company": "", "location": "", "salary": "",
            "posted": "", "deadline_raw": "", "employment_type": "",
            "desc": "", "url": full_url, "source": source,
        })
    return jobs


print("Helpers ready: HTTP with backoff, UK filtering, exclusions, JSON-LD reader.")


In [ ]:
# ── ELIGIBILITY AND DEADLINE READING ─────────────────────────────────────────
# The single most important filter in this notebook. A title match means nothing
# if the scheme is open only to penultimate-year undergraduates.
#
# Rule: never claim eligibility that the advert does not state. When the advert
# is silent, the verdict is "Unclear" and the row is kept for manual checking.
from datetime import date, datetime

# Riya is enrolled on a one-year MSc running to September 2026.
CANDIDATE_STUDY_LEVEL = "Postgraduate (MSc, enrolled to Sept 2026)"
CANDIDATE_GRADUATION_YEAR = 2026

UNDERGRAD_ONLY_SIGNALS = [
    "undergraduate students only", "open to undergraduates", "undergraduates only",
    "must be an undergraduate", "currently an undergraduate", "penultimate year",
    "penultimate-year", "second year student", "first year student",
    "1st year student", "2nd year student", "year in industry",
    "sandwich placement", "sandwich year", "bachelor's students",
    "bachelors students", "undergraduate degree programme",
]

POSTGRAD_SIGNALS = [
    "postgraduate", "post-graduate", "master's", "masters student", "msc",
    "ma student", "mba", "phd", "doctoral", "all degree levels",
    "undergraduate and postgraduate", "any level of study", "graduate students",
]

MBA_ONLY_SIGNALS = [
    "mba students only", "mba candidates only", "currently enrolled in an mba",
    "mba internship", "mba summer associate",
]

GRADUATE_EARLY_CAREER_SIGNALS = [
    "recent graduate", "recent graduates", "early careers", "early career",
    "graduate programme", "graduate program", "graduates welcome",
    "open to graduates", "no experience required", "career changer",
    "career changers", "returner",
]

CURRENT_STUDENT_SIGNALS = [
    "currently enrolled", "must be studying", "currently studying",
    "returning to university", "returning to full-time study",
    "must be a student", "enrolled at a university", "in full-time education",
]

NO_SPONSORSHIP_SIGNALS = [
    "no sponsorship", "cannot sponsor", "unable to sponsor", "will not sponsor",
    "does not sponsor", "do not sponsor", "without sponsorship",
    "sponsorship is not available", "must have the right to work",
    "right to work in the uk", "existing right to work",
    "eligible to work in the uk",
]

SPONSORSHIP_FRIENDLY_SIGNALS = [
    "visa sponsorship", "sponsorship available", "skilled worker",
    "sponsor licence", "graduate visa", "we sponsor",
]

_MONTHS = {
    "january": 1, "jan": 1, "february": 2, "feb": 2, "march": 3, "mar": 3,
    "april": 4, "apr": 4, "may": 5, "june": 6, "jun": 6, "july": 7, "jul": 7,
    "august": 8, "aug": 8, "september": 9, "sep": 9, "sept": 9, "october": 10,
    "oct": 10, "november": 11, "nov": 11, "december": 12, "dec": 12,
}


def parse_date_any(text):
    """Reads the common UK date spellings. Returns a date or None - never a guess."""
    if not text:
        return None
    s = str(text).strip()
    patterns = [
        (r"(\d{4})-(\d{1,2})-(\d{1,2})", lambda m: (int(m.group(1)), int(m.group(2)), int(m.group(3)))),
        (r"(\d{1,2})/(\d{1,2})/(\d{4})", lambda m: (int(m.group(3)), int(m.group(2)), int(m.group(1)))),
        # UK adverts routinely write "1st March 2027", so the ordinal is optional.
        (r"(\d{1,2})(?:st|nd|rd|th)?[\s\-]+([A-Za-z]{3,9})[\s\-,]+(\d{4})",
         lambda m: (int(m.group(3)), _MONTHS.get(m.group(2).lower(), 0), int(m.group(1)))),
        (r"([A-Za-z]{3,9})[\s\-]+(\d{1,2})(?:st|nd|rd|th)?[\s\-,]+(\d{4})",
         lambda m: (int(m.group(3)), _MONTHS.get(m.group(1).lower(), 0), int(m.group(2)))),
    ]
    for pattern, builder in patterns:
        match = re.search(pattern, s)
        if match:
            try:
                year, month, day = builder(match)
                if month and 1 <= month <= 12 and 1 <= day <= 31 and 2000 <= year <= 2100:
                    return date(year, month, day)
            except Exception:
                continue
    return None


def extract_deadline(text, jsonld_deadline=""):
    """Prefers the structured validThrough date, then wording in the advert."""
    structured = parse_date_any(jsonld_deadline)
    if structured:
        return structured, "structured data (validThrough)"
    body = str(text or "")
    cues = [
        r"(?:closing date|applications? close[sd]?|deadline|apply by|close[sd]? on|"
        r"last date to apply|applications? must be received by)\s*[:\-]?\s*([^\n\.;]{4,40})",
    ]
    for cue in cues:
        for match in re.finditer(cue, body, flags=re.IGNORECASE):
            parsed = parse_date_any(match.group(1))
            if parsed:
                return parsed, f"advert wording: '{match.group(0)[:60].strip()}'"
    return None, ""


def extract_start_date(text):
    body = str(text or "")
    cues = [
        r"(?:start date|starting|commenc\w+|begins?)\s*[:\-]?\s*([^\n\.;]{4,40})",
        r"\b((?:summer|spring|autumn|winter)\s+20\d{2})\b",
        r"\b((?:january|february|march|april|may|june|july|august|september|october|november|december)\s+20\d{2})\b",
    ]
    for cue in cues:
        match = re.search(cue, body, flags=re.IGNORECASE)
        if match:
            candidate = match.group(1).strip()
            parsed = parse_date_any(candidate)
            return parsed.isoformat() if parsed else candidate[:40]
    return ""


def extract_duration(text):
    match = re.search(
        r"\b(\d{1,2}|one|two|three|four|six|eight|nine|ten|twelve)[\s-]*"
        r"(week|month)s?\b(?![^\.]{0,30}notice)",
        str(text or ""), flags=re.IGNORECASE)
    if not match:
        return ""
    return f"{match.group(1)} {match.group(2)}s".replace("  ", " ")


def assess_eligibility(title, description):
    """
    Returns a dict describing what the advert actually says about who may apply.
    Nothing here infers eligibility from silence.
    """
    text = normalise_text(f"{title} {description}")
    notes = []

    # "undergraduate" ends in "graduate", so "undergraduate students only" would
    # otherwise read as the postgraduate-friendly "graduate students", and an
    # undergrad-only scheme would be presented as open to her. Mask it first.
    masked = text.replace("undergraduates", " ug ").replace("undergraduate", " ug ")

    undergrad = term_hits(text, UNDERGRAD_ONLY_SIGNALS)
    postgrad = term_hits(masked, POSTGRAD_SIGNALS)
    mba_only = term_hits(masked, MBA_ONLY_SIGNALS)
    graduate_ok = term_hits(masked, GRADUATE_EARLY_CAREER_SIGNALS)
    current_student = term_hits(text, CURRENT_STUDENT_SIGNALS)

    required_year = None
    year_match = re.search(r"graduat\w*\s+(?:in|by|during)?\s*(20\d{2})", text)
    if not year_match:
        year_match = re.search(r"class of\s+(20\d{2})", text)
    if year_match:
        required_year = int(year_match.group(1))

    # Verdict. Blocking means "do not present this as a top match".
    blocking = False
    if mba_only:
        # An MSc is not an MBA, so an MBA-only scheme is closed to this CV.
        verdict = "Likely ineligible - MBA-only scheme"
        blocking = True
        notes.append(f"advert says: {mba_only[0]}")
    elif undergrad and not postgrad:
        verdict = "Likely ineligible - undergraduate-only"
        blocking = True
        notes.append(f"advert says: {undergrad[0]}")
    elif postgrad:
        verdict = "Eligible - postgraduate / MSc accepted"
        notes.append(f"advert says: {postgrad[0]}")
    elif graduate_ok:
        verdict = "Eligible - open to graduates / early career"
        notes.append(f"advert says: {graduate_ok[0]}")
    elif current_student:
        verdict = "Eligible - current student status held (MSc to Sept 2026)"
        notes.append(f"advert says: {current_student[0]}")
    else:
        verdict = "Unclear"
        notes.append("advert does not state the study level required")

    if undergrad and postgrad:
        verdict = "Check - mentions both undergraduate and postgraduate study"
        notes.append("mixed study-level wording; read the advert before applying")

    if required_year and required_year != CANDIDATE_GRADUATION_YEAR:
        notes.append(f"advert asks for graduation in {required_year}; "
                     f"CV shows MSc completion in {CANDIDATE_GRADUATION_YEAR}")
        # Only downgrade to "Check". A year mismatch must never clear a block
        # already established by undergraduate-only or MBA-only wording.
        if not blocking:
            verdict = f"Check - advert asks for graduation in {required_year}"

    # Right to work. The CV states no visa status, so this is reported, never judged.
    if term_hits(text, NO_SPONSORSHIP_SIGNALS):
        work_note = "States UK right to work required / no sponsorship - check before applying"
    elif term_hits(text, SPONSORSHIP_FRIENDLY_SIGNALS):
        work_note = "Mentions sponsorship or visa support"
    else:
        work_note = "Not stated"

    degree_match = re.search(
        r"(stem|computer science|mathematics|statistics|economics|finance|business|engineering)\s+degree", text)
    if degree_match:
        notes.append(f"asks for a {degree_match.group(1)} degree")

    return {
        "eligibility": verdict,
        "eligibility_blocking": blocking,
        "eligibility_notes": "; ".join(notes[:4]),
        "required_graduation_year": required_year or "",
        "right_to_work": work_note,
    }


def deadline_status(deadline_value, today=None):
    """Closing Soon / Open / Deadline Unknown / Closed."""
    today = today or date.today()
    if not deadline_value:
        return "Deadline Unknown", 9999
    days_left = (deadline_value - today).days
    if days_left < 0:
        return "Closed", days_left
    if days_left <= CLOSING_SOON_DAYS:
        return "Closing Soon", days_left
    return "Open", days_left


def internship_type(title, description, employment_type=""):
    """Plain-language label for the Internship Type column."""
    text = normalise_text(f"{title} {description[:600]} {employment_type}")
    if "spring week" in text or "insight" in text:
        return "Insight / Spring programme"
    if "summer analyst" in text or "summer associate" in text or "summer intern" in text or "summer programme" in text:
        return "Summer internship"
    if "industrial placement" in text or "year in industry" in text or "sandwich" in text:
        return "Industrial placement (12 months)"
    if "placement" in text:
        return "Placement"
    if "graduate" in text and "intern" not in text:
        return "Graduate / early career"
    if "intern" in text:
        return "Internship"
    return "Internship (unspecified)"


print("Eligibility and deadline readers ready.")
print(f"  Candidate study level: {CANDIDATE_STUDY_LEVEL}")
print("  Verdicts: Eligible / Likely ineligible / Check / Unclear (never inferred from silence).")


In [ ]:
# ── MATCH SCORING (0-100) ────────────────────────────────────────────────────
# Transparent and additive: every row carries its own breakdown in the workbook.
# The score measures FIT ONLY. Eligibility is handled separately and is never
# overridden by a high score - an undergraduate-only scheme cannot reach the
# Top Matches sheet however well the title reads.

SCORE_WEIGHTS = {
    "Role relevance": 30,
    "CV skill match": 12,
    "Financial services": 10,
    "Data / analytics": 10,
    "Business analysis": 8,
    "Project / PMO": 8,
    "Education eligibility": 12,
    "Experience fit": 5,
    "Location": 5,
    "Internship suitability": 10,
}

FS_TERMS = [
    "bank", "banking", "financial services", "capital markets", "asset management",
    "wealth management", "insurance", "payments", "fintech", "credit", "lending",
    "investment", "treasury", "regulatory", "risk", "financial institution",
]

DATA_TERMS = [
    "data", "analytics", "analysis", "sql", "reporting", "dashboard", "insight",
    "power bi", "tableau", "excel", "python", "visualisation", "visualization",
    "metrics", "kpi", "data quality", "data governance",
]

BA_TERMS = [
    "business analysis", "business analyst", "requirements", "process", "stakeholder",
    "documentation", "workshops", "process improvement", "uat", "gap analysis",
]

PMO_TERMS = [
    "project", "programme", "program", "pmo", "milestone", "delivery", "raid",
    "governance", "planning", "jira", "agile", "waterfall", "prince2", "portfolio",
]


def classify_role_category(title, description=""):
    """Best-matching category, preferring a match in the title."""
    t = normalise_text(title)
    d = normalise_text(description)
    best = ("Other", 0, False)
    for category, terms in ROLE_CATEGORIES.items():
        title_hits = [term for term in terms if term in t]
        desc_hits = [term for term in terms if term in d]
        if title_hits:
            score = max(len(term) for term in title_hits) + 100
            in_title = True
        elif desc_hits:
            score = max(len(term) for term in desc_hits)
            in_title = False
        else:
            continue
        if score > best[1]:
            best = (category, score, in_title)
    return best[0], best[2]


def matching_skills(text):
    """Skills the CV evidences that this advert actually asks for."""
    found = []
    for skill, terms in CV_SKILLS.items():
        if any(term in text for term in terms):
            found.append(skill)
    return found


def missing_requirements(text):
    """Things the advert asks for that the CV does not evidence."""
    gaps = []
    for label, terms in GAP_REQUIREMENTS.items():
        if any(term in text for term in terms):
            gaps.append(label)
    return gaps


def score_internship(job):
    """
    Returns (score, breakdown_dict, matching_skills, missing_requirements).
    job is the consolidated record: title, company, location, desc, eligibility...
    """
    title = job.get("title", "")
    description = job.get("desc", "")
    t = normalise_text(title)
    text = normalise_text(f"{title} {description}")
    breakdown = {}

    # 1. Role relevance - a category hit in the title is worth far more.
    category, in_title = classify_role_category(title, description)
    if category == "Other":
        breakdown["Role relevance"] = 0
    elif in_title:
        breakdown["Role relevance"] = 30
    else:
        breakdown["Role relevance"] = 14

    # 2. CV skill match
    skills = matching_skills(text)
    breakdown["CV skill match"] = min(len(skills) * 3, SCORE_WEIGHTS["CV skill match"])

    # 3-6. Domain dimensions
    breakdown["Financial services"] = min(len(term_hits(text, FS_TERMS)) * 2.5, SCORE_WEIGHTS["Financial services"])
    breakdown["Data / analytics"] = min(len(term_hits(text, DATA_TERMS)) * 2, SCORE_WEIGHTS["Data / analytics"])
    breakdown["Business analysis"] = min(len(term_hits(text, BA_TERMS)) * 2, SCORE_WEIGHTS["Business analysis"])
    breakdown["Project / PMO"] = min(len(term_hits(text, PMO_TERMS)) * 2, SCORE_WEIGHTS["Project / PMO"])

    # 7. Education eligibility - scored, and separately enforced as a gate.
    eligibility = job.get("eligibility", "Unclear")
    if eligibility.startswith("Eligible"):
        breakdown["Education eligibility"] = 12
    elif eligibility.startswith("Check"):
        breakdown["Education eligibility"] = 6
    elif eligibility.startswith("Likely ineligible"):
        breakdown["Education eligibility"] = 0
    else:
        breakdown["Education eligibility"] = 5   # Unclear: neither rewarded nor punished

    # 8. Experience requirements - internships asking for years of experience are odd,
    #    but Riya has ~4 years, so a modest ask is a positive rather than a barrier.
    years = [int(y) for y in re.findall(r"(\d{1,2})\s*\+?\s*(?:years|yrs)", text)]
    years = [y for y in years if 0 < y <= 20]
    if not years:
        breakdown["Experience fit"] = 4
    elif min(years) <= 4:
        breakdown["Experience fit"] = 5
    else:
        breakdown["Experience fit"] = 1

    # 9. Location
    rank, _ = location_priority(job.get("location", ""))
    breakdown["Location"] = {1: 5, 2: 4, 3: 4, 4: 2}.get(rank, 2)

    # 10. Internship suitability
    if any(signal in t for signal in INTERNSHIP_TITLE_SIGNALS):
        breakdown["Internship suitability"] = 10
    elif is_internship(title, description, job.get("employment_type", "")):
        breakdown["Internship suitability"] = 6
    else:
        breakdown["Internship suitability"] = 0

    # Penalties for depth the CV does not evidence.
    unsupported = term_hits(text, UNSUPPORTED_TECH_TERMS)
    penalty = -10 if len(unsupported) >= 2 else (-4 if unsupported else 0)
    if penalty:
        breakdown["Unsupported tech penalty"] = penalty

    gaps = missing_requirements(text)
    score = int(max(0, min(100, round(sum(breakdown.values())))))
    return score, breakdown, skills, gaps


def format_breakdown(breakdown):
    return "; ".join(f"{name} {value:g}/{SCORE_WEIGHTS.get(name, abs(value)):g}"
                     if name in SCORE_WEIGHTS else f"{name} {value:g}"
                     for name, value in breakdown.items())


print("Scoring ready: 10 dimensions to 100, eligibility scored AND enforced separately.")


In [ ]:
# ── SOURCE 1 & 2: ADZUNA + REED APIS ─────────────────────────────────────────
# Same clients as the main job-search notebook, pointed at internship keywords
# and run across London and the UK as a whole.
import pandas as pd

raw_jobs = []

# Reed returns a short summary in search results. Its per-job endpoint returns the
# full advert, which is what the workbook needs. Only fetched for postings that
# already look like internships, and capped so the notebook stays polite.
FETCH_FULL_REED_DESCRIPTIONS = True
REED_DETAIL_LIMIT = 120


def fetch_adzuna(keyword, location):
    jobs = []
    if not (ADZUNA_APP_ID and ADZUNA_APP_KEY):
        return jobs
    url = (
        f"https://api.adzuna.com/v1/api/jobs/gb/search/1"
        f"?app_id={ADZUNA_APP_ID}&app_key={ADZUNA_APP_KEY}"
        f"&results_per_page={RESULTS_PER_SOURCE}"
        f"&what={requests.utils.quote(keyword)}"
        f"&where={requests.utils.quote(location)}"
        f"&sort_by=date&content-type=application/json"
    )
    try:
        response = request_with_backoff(url, timeout=15)
        response.raise_for_status()
        for item in response.json().get("results", []):
            jobs.append({
                "title": item.get("title", ""),
                "company": (item.get("company") or {}).get("display_name", ""),
                "location": (item.get("location") or {}).get("display_name", ""),
                "salary": f"{item.get('salary_min', '') or ''}-{item.get('salary_max', '') or ''}".strip("-"),
                "posted": str(item.get("created", ""))[:10],
                "deadline_raw": "",
                "employment_type": item.get("contract_time", "") or "",
                "desc": clean_description(item.get("description", "")),
                "url": item.get("redirect_url", ""),
                "source": "Adzuna",
            })
    except Exception as error:
        print(f"    Adzuna [{keyword} / {location}]: {str(error)[:70]}")
    return jobs


def fetch_reed(keyword, location):
    jobs = []
    if not REED_API_KEY:
        return jobs
    url = (
        f"https://www.reed.co.uk/api/1.0/search"
        f"?keywords={requests.utils.quote(keyword)}"
        f"&locationName={requests.utils.quote(location)}"
        f"&resultsToTake={RESULTS_PER_SOURCE}"
    )
    try:
        response = request_with_backoff(url, auth=(REED_API_KEY, ""), timeout=15)
        response.raise_for_status()
        for item in response.json().get("results", []):
            jobs.append({
                "title": item.get("jobTitle", ""),
                "company": item.get("employerName", ""),
                "location": item.get("locationName", ""),
                "salary": f"{item.get('minimumSalary', '') or ''}-{item.get('maximumSalary', '') or ''}".strip("-"),
                "posted": str(item.get("date", ""))[:10],
                "deadline_raw": str(item.get("expirationDate", "") or "")[:10],
                "employment_type": "Part time" if item.get("partTime") else "Full time",
                "desc": clean_description(item.get("jobDescription", "")),
                "url": item.get("jobUrl", ""),
                "source": "Reed",
                "_reed_id": item.get("jobId", ""),
            })
    except Exception as error:
        print(f"    Reed [{keyword} / {location}]: {str(error)[:70]}")
    return jobs


def enrich_reed_descriptions(jobs):
    """Swaps Reed's search-result summary for the full advert, where available."""
    if not (FETCH_FULL_REED_DESCRIPTIONS and REED_API_KEY):
        return 0
    enriched = 0
    targets = [j for j in jobs if j.get("source") == "Reed" and j.get("_reed_id")][:REED_DETAIL_LIMIT]
    for job in targets:
        try:
            response = request_with_backoff(
                f"https://www.reed.co.uk/api/1.0/jobs/{job['_reed_id']}",
                auth=(REED_API_KEY, ""), timeout=15)
            if response.status_code != 200:
                continue
            payload = response.json()
            full = clean_description(payload.get("jobDescription", ""))
            if len(full) > len(job.get("desc", "")):
                job["desc"] = full
                enriched += 1
            job["deadline_raw"] = job.get("deadline_raw") or str(payload.get("expirationDate", "") or "")[:10]
        except Exception:
            continue
        time.sleep(0.2)
    return enriched


print("Searching Adzuna and Reed for internships...")
print("-" * 68)
adzuna_total = reed_total = 0
for location in SEARCH_LOCATIONS:
    for keyword in INTERNSHIP_SEARCH_KEYWORDS:
        found_a = fetch_adzuna(keyword, location)
        found_r = fetch_reed(keyword, location)
        raw_jobs += found_a + found_r
        adzuna_total += len(found_a)
        reed_total += len(found_r)
        time.sleep(0.2)
    print(f"  {location:<20} running total: Adzuna {adzuna_total}, Reed {reed_total}")

if not (ADZUNA_APP_ID and ADZUNA_APP_KEY):
    print("  Adzuna skipped: no API key set in the settings cell.")
if not REED_API_KEY:
    print("  Reed skipped: no API key set in the settings cell.")

enriched = enrich_reed_descriptions(raw_jobs)
print(f"  Reed full descriptions fetched: {enriched}")

note_source("Adzuna API", adzuna_total, "keyed API")
note_source("Reed API", reed_total, f"keyed API, {enriched} full JDs")
print("-" * 68)
print(f"Job-board API rows so far: {len(raw_jobs)}")


In [ ]:
# ── SOURCE 3: LINKEDIN ───────────────────────────────────────────────────────
# Same public-page reader as the main notebook, with LinkedIn's own internship
# experience-level filter applied (f_E=1) and the guest card endpoint tried
# first because it returns markup that changes less often.
LINKEDIN_SEARCHES = [
    "data analyst intern", "data analytics intern", "business intelligence intern",
    "data governance intern", "data quality intern", "business analyst intern",
    "risk analytics intern", "regulatory reporting intern", "financial data intern",
    "project management intern", "pmo intern", "strategy intern",
    "business operations intern", "consulting intern", "summer analyst",
    "summer associate", "industrial placement data", "data placement",
]

LINKEDIN_LOCATIONS = ["London, United Kingdom", "United Kingdom"]

linkedin_jobs = []
seen_linkedin = set()


def parse_linkedin_cards(html):
    cards_found = []
    soup = BeautifulSoup(html, "html.parser")
    cards = soup.find_all("div", class_=lambda x: x and "base-card" in str(x))
    if not cards:
        cards = soup.find_all("li")
    for card in cards:
        title_tag = card.find(["h3", "h2"], class_=lambda x: x and "title" in str(x).lower())
        company_tag = card.find(["h4", "a"], class_=lambda x: x and "subtitle" in str(x).lower())
        loc_tag = card.find("span", class_=lambda x: x and "location" in str(x).lower())
        link_tag = card.find("a", class_=lambda x: x and "full-link" in str(x).lower())
        if link_tag is None:
            link_tag = card.find("a", href=True)
        title = title_tag.get_text(strip=True) if title_tag else ""
        company = company_tag.get_text(strip=True) if company_tag else ""
        location = loc_tag.get_text(strip=True) if loc_tag else ""
        link = link_tag["href"].split("?")[0] if link_tag and link_tag.get("href") else ""
        if title and link:
            cards_found.append((title, company, location, link))
    return cards_found


print("Reading LinkedIn public job pages...")
print("-" * 68)
for location in LINKEDIN_LOCATIONS:
    for keyword in LINKEDIN_SEARCHES:
        encoded_kw = requests.utils.quote(keyword)
        encoded_loc = requests.utils.quote(location)
        urls = [
            # Guest card endpoint: returns just the result cards.
            f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search"
            f"?keywords={encoded_kw}&location={encoded_loc}&f_E=1&start=0",
            # Full search page as a fallback.
            f"https://www.linkedin.com/jobs/search/?keywords={encoded_kw}"
            f"&location={encoded_loc}&f_TPR=r2592000&f_E=1",
        ]
        found = 0
        for url in urls:
            try:
                response = request_with_backoff(url, headers=HEADERS_BROWSER, timeout=20)
                if response.status_code != 200 or len(response.text) < 200:
                    continue
                for title, company, loc_text, link in parse_linkedin_cards(response.text):
                    if link in seen_linkedin:
                        continue
                    seen_linkedin.add(link)
                    linkedin_jobs.append({
                        "title": title, "company": company,
                        "location": loc_text or location,
                        "salary": "", "posted": "", "deadline_raw": "",
                        "employment_type": "Internship",
                        "desc": "",          # LinkedIn guest pages do not expose the body
                        "url": link, "source": "LinkedIn",
                    })
                    found += 1
                if found:
                    break
            except Exception:
                continue
        time.sleep(1.5)
    print(f"  {location:<26} running total: {len(linkedin_jobs)}")

raw_jobs += linkedin_jobs
note_source("LinkedIn", len(linkedin_jobs), "public pages; titles only, no description")
print("-" * 68)
print(f"Rows so far: {len(raw_jobs)}")


In [ ]:
# ── SOURCE 4: STUDENT AND GRADUATE JOB BOARDS ────────────────────────────────
# None of these publish an API, so this reads their public search pages:
#   1. schema.org JobPosting markup (JSON-LD) where the site embeds it - this is
#      the good case, because it carries the description AND the closing date;
#   2. otherwise, job links harvested from the page, giving a title-only row.
# Every board is independent: one failing never stops the others, and each
# reports its own outcome in the source summary.
#
# These sites change their markup and several use bot protection, so treat a
# zero here as "check the site by hand", not as "there are no internships".

BOARD_KEYWORDS = [
    "data analyst intern", "data analytics intern", "business analyst intern",
    "business intelligence intern", "risk intern", "finance intern",
    "project management intern", "strategy intern",
]

JOB_BOARDS = {
    "Bright Network": {
        "templates": [
            "https://www.brightnetwork.co.uk/search/?search={kw}",
            "https://www.brightnetwork.co.uk/graduate-jobs/search/?search={kw}",
        ],
        "hints": ("/graduate-jobs/", "/internships/", "/work-experience/", "/job"),
    },
    "RateMyPlacement": {
        "templates": [
            "https://www.ratemyplacement.co.uk/search-jobs/?search={kw}",
            "https://www.ratemyplacement.co.uk/search-jobs?q={kw}",
        ],
        "hints": ("/job/", "/placement", "/internship", "/jobs/"),
    },
    "TargetJobs": {
        "templates": [
            "https://targetjobs.co.uk/search-jobs?keywords={kw}",
            "https://targetjobs.co.uk/jobs?search_api_fulltext={kw}",
        ],
        "hints": ("/job/", "/jobs/", "/internship", "/placement"),
    },
    "Milkround": {
        "templates": [
            "https://www.milkround.com/searchjobs/?Keywords={kw}",
            "https://www.milkround.com/jobs/{slug}/",
        ],
        "hints": ("/job/", "/jobs/", "/internship"),
    },
    "Graduate-Jobs": {
        "templates": [
            "https://www.graduate-jobs.com/search?q={kw}",
            "https://www.graduate-jobs.com/jobs/{slug}",
        ],
        "hints": ("/job/", "/jobs/", "/vacancy"),
    },
    "Indeed UK": {
        # Indeed applies bot protection to its search pages and frequently returns
        # a challenge instead of results. Left in because it sometimes works.
        "templates": [
            "https://uk.indeed.com/jobs?q={kw}&l=London&fromage=30",
            "https://uk.indeed.com/jobs?q={kw}&l=United+Kingdom&fromage=30",
        ],
        "hints": ("/viewjob", "/rc/clk", "/job/"),
    },
}


def fetch_board(name, config, keywords):
    """One board, every keyword. Returns rows and a short status note."""
    collected = []
    seen = set()
    structured = 0
    blocked = 0
    for keyword in keywords:
        encoded = requests.utils.quote(keyword)
        slug = keyword.replace(" ", "-")
        for template in config["templates"]:
            url = template.format(kw=encoded, slug=slug)
            try:
                response = request_with_backoff(url, headers=HEADERS_BROWSER, timeout=20, retries=2)
                if response.status_code in (403, 429) or len(response.text) < 500:
                    blocked += 1
                    continue
                if response.status_code != 200:
                    continue
                found = parse_jsonld_jobs(response.text, name, page_url=url)
                if found:
                    structured += len(found)
                else:
                    found = harvest_job_links(response.text, url, name, config["hints"])
                for job in found:
                    key = job.get("url") or job.get("title")
                    if not key or key in seen:
                        continue
                    seen.add(key)
                    collected.append(job)
                if found:
                    break
            except Exception:
                continue
            finally:
                time.sleep(1.2)
    detail = f"{structured} with structured data" if structured else (
        "blocked or challenged" if blocked else "no structured data found")
    return collected, detail


print("Reading student and graduate job boards...")
print("-" * 68)
board_rows = 0
for board_name, board_config in JOB_BOARDS.items():
    try:
        rows, detail = fetch_board(board_name, board_config, BOARD_KEYWORDS)
    except Exception as error:
        rows, detail = [], f"failed: {str(error)[:50]}"
    raw_jobs += rows
    board_rows += len(rows)
    note_source(board_name, len(rows), detail)

print("-" * 68)
print(f"Board rows added: {board_rows}")
print(f"Rows so far: {len(raw_jobs)}")


In [ ]:
# ── SOURCE 5: EMPLOYER CAREER PAGES ──────────────────────────────────────────
# Reuses the Greenhouse and Lever readers from the main notebook. Both expose a
# public JSON API, so these are the most reliable non-keyed sources here and the
# ones that give a job-specific application URL plus the full advert body.
# Employers chosen for relevance to this CV: banks, financial-data firms,
# consultancies, regtech and fintech.
GREENHOUSE_BOARDS = {
    "Quantexa": "quantexa",
    "ComplyAdvantage": "complyadvantage",
    "Monzo": "monzo",
    "GoCardless": "gocardless",
    "Starling Bank": "starlingbank",
    "Thought Machine": "thoughtmachine",
    "Checkout.com": "checkoutdotcom",
    "Klarna": "klarna",
    "Experian": "experian",
    "Capgemini": "capgemini",
    "Slalom": "slalom",
    "PA Consulting": "paconsulting",
    "Tide": "tide",
    "Funding Circle": "fundingcircle",
    "Iwoca": "iwoca",
    "Zopa": "zopabank",
    "TrueLayer": "truelayer",
    "SumUp": "sumup",
}

LEVER_BOARDS = {
    "OakNorth": "oaknorth.ai",
    "Moneybox": "moneyboxapp",
    "Allica Bank": "allica-bank",
    "Zego": "zego",
    "Cleo": "meetcleo",
}

employer_jobs = []
print("Reading employer career pages...")
print("-" * 68)

greenhouse_found = 0
for company, token in GREENHOUSE_BOARDS.items():
    try:
        response = request_with_backoff(
            f"https://boards-api.greenhouse.io/v1/boards/{token}/jobs?content=true",
            headers=HEADERS_BROWSER, timeout=20, retries=2)
        if response.status_code != 200:
            continue
        for job in response.json().get("jobs", []):
            title = job.get("title", "")
            offices = job.get("offices", [])
            location = offices[0].get("name", "") if offices else (job.get("location") or {}).get("name", "")
            if not is_internship(title, "", (job.get("categories") or {}).get("commitment", "")):
                continue
            if not is_uk_location(location):
                continue
            employer_jobs.append({
                "title": title, "company": company, "location": location or "UK",
                "salary": "", "posted": str(job.get("updated_at", ""))[:10],
                "deadline_raw": "", "employment_type": "",
                "desc": clean_description(job.get("content", "")),
                "url": job.get("absolute_url", ""), "source": "Employer careers (Greenhouse)",
            })
            greenhouse_found += 1
    except Exception:
        continue
    finally:
        time.sleep(0.3)
note_source("Greenhouse employer boards", greenhouse_found, f"{len(GREENHOUSE_BOARDS)} employers checked")

lever_found = 0
for company, slug in LEVER_BOARDS.items():
    try:
        response = request_with_backoff(
            f"https://api.lever.co/v0/postings/{slug}?mode=json",
            headers=HEADERS_BROWSER, timeout=20, retries=2)
        if response.status_code != 200:
            continue
        for job in response.json():
            title = job.get("text", "")
            location = (job.get("categories") or {}).get("location", "")
            if not is_internship(title):
                continue
            if not is_uk_location(location):
                continue
            employer_jobs.append({
                "title": title, "company": company, "location": location or "UK",
                "salary": "", "posted": "", "deadline_raw": "",
                "employment_type": (job.get("categories") or {}).get("commitment", ""),
                "desc": clean_description(job.get("descriptionPlain") or job.get("description") or ""),
                "url": job.get("hostedUrl", ""), "source": "Employer careers (Lever)",
            })
            lever_found += 1
    except Exception:
        continue
    finally:
        time.sleep(0.3)
note_source("Lever employer boards", lever_found, f"{len(LEVER_BOARDS)} employers checked")

raw_jobs += employer_jobs
print("-" * 68)
print(f"Employer rows added: {len(employer_jobs)}")
print(f"Total collected rows: {len(raw_jobs)}")


In [ ]:
# ── CONSOLIDATE: FILTER, ASSESS, SCORE, DEDUPLICATE ──────────────────────────
from collections import Counter

records = []
rejected = Counter()

for job in raw_jobs:
    title = (job.get("title") or "").strip()
    description = job.get("desc") or ""
    if not title:
        rejected["no title"] += 1
        continue

    # Internship-only tool: a permanent role never belongs here.
    if not is_internship(title, description, job.get("employment_type", "")):
        rejected["not an internship"] += 1
        continue

    reason = exclusion_reason(title, description)
    if reason:
        rejected[reason.split(" - ")[-1][:30]] += 1
        continue

    location = job.get("location") or ""
    if not is_uk_location(location):
        rejected["outside the UK"] += 1
        continue

    eligibility = assess_eligibility(title, description)
    deadline_date, deadline_source = extract_deadline(description, job.get("deadline_raw", ""))
    status, days_left = deadline_status(deadline_date)
    _, location_type = location_priority(location)

    record = {
        "Job Title": title,
        "Company": (job.get("company") or "").strip() or "Not stated",
        "Location": location or "Not stated",
        "Location Type": location_type,
        "Internship Type": internship_type(title, description, job.get("employment_type", "")),
        "Source": job.get("source", ""),
        "Duplicate Sources": "",
        "Salary": job.get("salary", "") or "Not stated",
        "Posted": job.get("posted", ""),
        "Start Date": extract_start_date(description),
        "Duration": extract_duration(description),
        "Deadline": deadline_date.isoformat() if deadline_date else "",
        "Deadline Status": status,
        "Days Left": days_left if deadline_date else "",
        "Deadline Source": deadline_source,
        "Eligibility": eligibility["eligibility"],
        "Eligibility Notes": eligibility["eligibility_notes"],
        "Right to Work": eligibility["right_to_work"],
        "_eligibility_blocking": eligibility["eligibility_blocking"],
        "Job Description": description,
        "Apply Link": job.get("url", "") or "",
    }

    scoring_input = dict(record)
    scoring_input.update({"title": title, "desc": description, "location": location,
                          "employment_type": job.get("employment_type", ""),
                          "eligibility": record["Eligibility"]})
    score, breakdown, skills, gaps = score_internship(scoring_input)

    record.update({
        "Match Score": score,
        "Score Breakdown": format_breakdown(breakdown),
        "Role Category": classify_role_category(title, description)[0],
        "Key Matching Skills": ", ".join(skills) if skills else "None identifiable from the advert text",
        "Missing Requirements": ", ".join(gaps) if gaps else "None identified",
        "Description Quality": (
            "Full advert" if len(description) > 400 else
            "Partial / summary" if description.strip() else "Title only"
        ),
        "AI Assessment": "",
        "AI Key Matching Skills": "",
        "AI Missing Requirements": "",
        "AI Eligibility Concerns": "",
        "AI Worth Applying": "",
    })
    records.append(record)

print("Filtering summary")
print("-" * 68)
print(f"  Collected rows:        {len(raw_jobs)}")
for label, count in rejected.most_common():
    print(f"  Rejected - {label:<28} {count}")
print(f"  Internship candidates: {len(records)}")

# ── DEDUPLICATION ────────────────────────────────────────────────────────────
# The same internship appears on LinkedIn, Bright Network and Adzuna at once.
# Keep one row: fullest description first, then the most job-specific URL, then
# the more reliable source. The losing copy still contributes its source name and
# fills any field the winner left blank.
SOURCE_RANK = {
    "Employer careers (Greenhouse)": 5, "Employer careers (Lever)": 5,
    "Reed": 4, "Adzuna": 4, "Bright Network": 3, "RateMyPlacement": 3,
    "TargetJobs": 3, "Milkround": 3, "Graduate-Jobs": 3, "Indeed UK": 2,
    "LinkedIn": 1,
}


def url_specificity(url):
    u = normalise_text(url)
    if not u:
        return 0
    if any(token in u for token in ["/job/", "/jobs/", "jobid", "job_id", "requisition",
                                    "/vacancy", "/postings/", "viewjob", "-job-", "/internship"]):
        return 3
    return 2 if u.rstrip("/").count("/") > 3 else 1


def record_quality(record):
    return (
        len(record.get("Job Description", "") or ""),
        url_specificity(record.get("Apply Link", "")),
        SOURCE_RANK.get(record.get("Source", ""), 1),
    )


def dedup_key(record):
    title = re.sub(r"[^a-z0-9 ]", "", normalise_text(record["Job Title"]))
    company = re.sub(r"[^a-z0-9 ]", "", normalise_text(record["Company"]))
    return (title, company)


best_by_key = {}
sources_by_key = {}
for record in records:
    key = dedup_key(record)
    sources_by_key.setdefault(key, set()).add(record.get("Source", ""))
    incumbent = best_by_key.get(key)
    if incumbent is None:
        best_by_key[key] = record
        continue
    winner, loser = ((record, incumbent) if record_quality(record) > record_quality(incumbent)
                     else (incumbent, record))
    for field in ("Salary", "Posted", "Start Date", "Duration", "Deadline",
                  "Deadline Status", "Days Left", "Location", "Apply Link", "Job Description"):
        current = str(winner.get(field, "") or "").strip()
        if current in ("", "Not stated", "Deadline Unknown") and str(loser.get(field, "") or "").strip():
            winner[field] = loser[field]
    winner["Match Score"] = max(winner.get("Match Score", 0), loser.get("Match Score", 0))
    best_by_key[key] = winner

deduped = []
for key, record in best_by_key.items():
    others = sorted(s for s in sources_by_key.get(key, set()) if s and s != record.get("Source"))
    record["Duplicate Sources"] = ", ".join(others)
    deduped.append(record)

duplicates_removed = len(records) - len(deduped)
deduped.sort(key=lambda r: (-r.get("Match Score", 0), r.get("Days Left", 9999) or 9999))

active = [r for r in deduped if r["Deadline Status"] != "Closed"]
closed = [r for r in deduped if r["Deadline Status"] == "Closed"]
top_matches = [r for r in active
               if r["Match Score"] >= TOP_MATCH_MIN_SCORE and not r["_eligibility_blocking"]]
with_deadlines = sorted([r for r in active if r["Deadline"]],
                        key=lambda r: r.get("Days Left", 9999))

print()
print("Consolidation")
print("-" * 68)
print(f"  Duplicates merged:     {duplicates_removed}")
print(f"  Unique internships:    {len(deduped)}")
print(f"  Active (not closed):   {len(active)}")
print(f"  Closed, excluded:      {len(closed)}")
print(f"  Top matches:           {len(top_matches)}  (score >= {TOP_MATCH_MIN_SCORE} and not blocked on eligibility)")
print(f"  With a known deadline: {len(with_deadlines)}")
print()
print("Eligibility spread:")
for verdict, count in Counter(r["Eligibility"] for r in active).most_common():
    print(f"  {verdict[:52]:<54} {count}")
print()
print("Role categories:")
for category, count in Counter(r["Role Category"] for r in active).most_common():
    print(f"  {category:<40} {count}")


In [ ]:
# ── EXCEL EXPORT ─────────────────────────────────────────────────────────────
# Same writer as the main notebook: full job descriptions, real clickable
# hyperlinks, frozen headers, filters and wrapped text.
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from google.colab import files

# Excel refuses more than 32,767 characters in one cell. The advert is never
# trimmed: anything beyond the limit spills into a continuation column.
EXCEL_CELL_LIMIT = 32000
_ILLEGAL_XLSX_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


def clean_for_excel(value):
    if value is None:
        return ""
    return _ILLEGAL_XLSX_CHARS.sub(" ", str(value))


def split_description_for_excel(text):
    text = clean_for_excel(text)
    if len(text) <= EXCEL_CELL_LIMIT:
        return text, ""
    remainder = text[EXCEL_CELL_LIMIT:]
    if len(remainder) > EXCEL_CELL_LIMIT:
        remainder = remainder[:EXCEL_CELL_LIMIT - 60] + " [...continues - open the Apply Link]"
    return text[:EXCEL_CELL_LIMIT], remainder


MAIN_COLUMNS = [
    "Match Score", "Job Title", "Company", "Location", "Location Type",
    "Internship Type", "Role Category", "Source", "Duplicate Sources", "Salary",
    "Posted", "Start Date", "Duration", "Deadline", "Deadline Status", "Days Left",
    "Eligibility", "Eligibility Notes", "Right to Work", "Key Matching Skills",
    "Missing Requirements", "Score Breakdown", "Description Quality",
    "Job Description", "Job Description (continued)",
    "AI Assessment", "AI Key Matching Skills", "AI Missing Requirements",
    "AI Eligibility Concerns", "AI Worth Applying", "Apply Link",
]

HYPERLINK_COLUMNS = {"Apply Link"}
WRAP_COLUMNS = {
    "Job Description", "Job Description (continued)", "Eligibility Notes",
    "Key Matching Skills", "Missing Requirements", "Score Breakdown",
    "AI Assessment", "AI Key Matching Skills", "AI Missing Requirements",
    "AI Eligibility Concerns", "Notes",
}


def to_frame(rows, columns=None):
    columns = columns or MAIN_COLUMNS
    prepared = []
    for row in rows:
        item = {k: v for k, v in row.items() if not k.startswith("_")}
        main, overflow = split_description_for_excel(item.get("Job Description", ""))
        item["Job Description"] = main
        item["Job Description (continued)"] = overflow
        for column in columns:
            item.setdefault(column, "")
        prepared.append({column: clean_for_excel(item.get(column, "")) if isinstance(item.get(column), str)
                         else item.get(column, "") for column in columns})
    return pd.DataFrame(prepared, columns=columns)


def format_workbook(path):
    """Clickable Apply Links, frozen headers, filters, readable descriptions."""
    workbook = load_workbook(path)
    link_font = Font(color="0563C1", underline="single")
    header_font = Font(bold=True, color="FFFFFF")
    header_fill = PatternFill("solid", fgColor="2F5597")
    urgent_fill = PatternFill("solid", fgColor="FCE4E4")
    links_made = 0

    for worksheet in workbook.worksheets:
        if worksheet.max_row < 1:
            continue
        headers = [cell.value for cell in worksheet[1]]
        for cell in worksheet[1]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = Alignment(vertical="center")
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = f"A1:{get_column_letter(worksheet.max_column)}{worksheet.max_row}"

        has_wrapped = False
        for index, name in enumerate(headers, start=1):
            letter = get_column_letter(index)
            if name in HYPERLINK_COLUMNS:
                worksheet.column_dimensions[letter].width = 48
                for row in range(2, worksheet.max_row + 1):
                    cell = worksheet.cell(row=row, column=index)
                    url = str(cell.value or "").strip()
                    if url.lower().startswith(("http://", "https://")):
                        cell.hyperlink = url
                        cell.font = link_font
                        cell.alignment = Alignment(vertical="top")
                        links_made += 1
                    else:
                        # No usable URL: leave it blank rather than invent one.
                        cell.value = None
            elif name in WRAP_COLUMNS:
                has_wrapped = True
                worksheet.column_dimensions[letter].width = 70
                for row in range(2, worksheet.max_row + 1):
                    worksheet.cell(row=row, column=index).alignment = Alignment(wrap_text=True, vertical="top")
            else:
                worksheet.column_dimensions[letter].width = max(12, min(30, len(str(name or "")) + 6))

        # Flag the rows that close soonest.
        if "Deadline Status" in headers:
            status_col = headers.index("Deadline Status") + 1
            for row in range(2, worksheet.max_row + 1):
                if worksheet.cell(row=row, column=status_col).value == "Closing Soon":
                    for col in range(1, worksheet.max_column + 1):
                        worksheet.cell(row=row, column=col).fill = urgent_fill

        if has_wrapped:
            for row in range(2, worksheet.max_row + 1):
                worksheet.row_dimensions[row].height = 42

    workbook.save(path)
    return links_made


df_all = to_frame(active)
df_top = to_frame(top_matches)
df_deadlines = to_frame(with_deadlines, [
    "Deadline", "Days Left", "Deadline Status", "Job Title", "Company",
    "Internship Type", "Eligibility", "Match Score", "Deadline Source", "Apply Link",
])
df_tracker = pd.DataFrame([
    {"Company": r["Company"], "Role": r["Job Title"], "Apply Link": r["Apply Link"],
     "Deadline": r["Deadline"], "Status": "Not started", "Date Applied": "", "Notes": ""}
    for r in (top_matches or active)
], columns=["Company", "Role", "Apply Link", "Deadline", "Status", "Date Applied", "Notes"])

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    df_all.to_excel(writer, index=False, sheet_name="All Internships")
    df_top.to_excel(writer, index=False, sheet_name="Top Matches")
    df_deadlines.to_excel(writer, index=False, sheet_name="Deadline Tracker")
    df_tracker.to_excel(writer, index=False, sheet_name="Applied Tracker")

links = format_workbook(OUTPUT_FILE)

print("Workbook written")
print("-" * 68)
print(f"  All Internships:   {len(df_all)} rows")
print(f"  Top Matches:       {len(df_top)} rows")
print(f"  Deadline Tracker:  {len(df_deadlines)} rows")
print(f"  Applied Tracker:   {len(df_tracker)} rows")
print(f"  Clickable links:   {links}")
full_jds = sum(1 for r in active if len(r.get("Job Description", "") or "") > 400)
print(f"  Full adverts kept: {full_jds}/{len(active)} (stored complete, never trimmed)")
print()
print("Source summary")
print("-" * 68)
for name, info in SOURCE_STATUS.items():
    print(f"  {name:<30} {info['roles']:>4} roles   {info['detail']}")
print()
files.download(OUTPUT_FILE)
print(f"Downloaded {OUTPUT_FILE}")


In [ ]:
# ── OPTIONAL AI REVIEW (OPENROUTER) ──────────────────────────────────────────
# The ranking above is deterministic and does not call any model. Set
# RUN_AI_REVIEW = True, add OPENROUTER_API_KEY under Colab Secrets, and run this
# cell to have a model read the shortlist against the CV.
RUN_AI_REVIEW   = False
AI_REVIEW_LIMIT = 30      # None attempts every active row, in batches
AI_BATCH_SIZE   = 8
MODEL_NAME      = "openai/gpt-4.1-mini"
MAX_RETRIES     = 2
TEMPERATURE     = 0.2
MAX_TOKENS      = 3000

!pip install -U openai -q

from openai import OpenAI


def get_openrouter_api_key():
    return _secret("OPENROUTER_API_KEY", "")


# AI INPUT ONLY. This shortening exists to control token spend and is never
# applied to the Excel "Job Description" column, which keeps the full advert.
def trim_job_description(description, max_words=1200):
    text = re.sub(r"\s+", " ", str(description or "")).strip()
    if not text:
        return ""
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words])


def parse_ai_review_json(text):
    if not text:
        return []
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
        cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        parsed = json.loads(cleaned)
    except Exception:
        match = re.search(r"\[.*\]|\{.*\}", cleaned, flags=re.DOTALL)
        if not match:
            return []
        try:
            parsed = json.loads(match.group(0))
        except Exception:
            return []
    if isinstance(parsed, dict):
        parsed = parsed.get("results", [])
    return parsed if isinstance(parsed, list) else []


CANDIDATE_PROFILE = """
CANDIDATE: Riya Singh, based in the United Kingdom. Currently studying an MSc in
Programme and Project Management at the University of Warwick (Sept 2025 - Sept
2026), so she is an enrolled POSTGRADUATE student for the whole of this search.
Earlier: BTech Information Technology, SRM Institute (2017-2021).

WORK HISTORY (all of it):
1. Wells Fargo, Data Management Analyst, Feb 2024 - Jul 2025. Month-end
   reporting cycles and reconciliation of regulatory data across Commercial
   Banking. Managed FR Y-14 Schedule H1 and H2 submissions under Federal Reserve
   guidelines. Built bi-weekly, monthly and quarterly Data Quality packages.
   Data quality checks for FR Y-14Q and BCBS 239. Complex SQL over large
   multi-source financial datasets. Governance documentation, audit trails and
   45 audit-ready reports.
2. Mu Sigma, Business Analyst, Jul 2021 - Feb 2024. Helped build and implement a
   data governance framework: documentation standards, metadata management,
   information control procedures. Managed metadata repositories. Executive-level
   summaries from disparate datasets. Stakeholder management and client check-ins
   on machine learning models for purchase-propensity audiences - a delivery and
   stakeholder role, NOT model engineering.
3. Tata Consultancy Services, intern, Dec 2018 - Jan 2019. Exposure to financial
   and regulatory data frameworks; documented systems and controls for audits.
PROJECTS: led a go-to-market strategy team for an HVAC technology firm entering
Singapore (Warwick TeamWork programme); co-founded an AI beauty e-commerce
platform and won a UK-wide national start-up competition.

TOOLS: SQL, Python, Advanced Excel, VBA, Power BI, Tableau, Alteryx, SharePoint,
ServiceNow, JIRA, Trello.
CERTIFICATIONS: APM, Machine Learning, Data Analytics with Power BI.
PRINCE2 is listed as "familiar" only - not certified.

SHE IS NOT, and you must never assume otherwise: a software or data engineer, a
data scientist, a quantitative researcher, a qualified accountant or actuary, or
a people manager. The CV states no visa or right-to-work status and no salary
expectation - never infer either.

ELIGIBILITY MATTERS MORE THAN FIT. She is a postgraduate with roughly four years
of prior full-time experience. An internship restricted to penultimate-year or
other undergraduates, or to MBA students, is NOT open to her however well the
subject matter matches. Say so plainly when that is the case.
"""


def compact_job_for_ai(row, row_id):
    return {
        "row_id": row_id,
        "job_title": row.get("Job Title", ""),
        "company": row.get("Company", ""),
        "location": row.get("Location", ""),
        "internship_type": row.get("Internship Type", ""),
        "deadline": row.get("Deadline", ""),
        "rule_eligibility_reading": row.get("Eligibility", ""),
        # Trimmed copy for the model only; the workbook keeps the full advert.
        "cleaned_job_description": trim_job_description(row.get("Job Description", "")),
    }


class AIProvider:
    def __init__(self, api_key, model_name=MODEL_NAME):
        self.client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
        self.model_name = model_name

    def review(self, jobs, candidate_profile):
        prompt = f"""
You are reviewing UK internships for this candidate:
{candidate_profile}

For each internship below, judge it using ONLY the candidate profile above and
the job text provided. Never invent experience, qualifications or visa status.
If the advert does not state who may apply, say the eligibility is unclear
rather than assuming she qualifies.

Return ONLY a valid JSON array. No markdown, no commentary. Each item must have:
  row_id                 - the id given
  what_the_role_involves - one sentence on the actual work
  why_it_matches         - one or two sentences tying it to her real experience
  key_matching_skills    - comma-separated, only skills evidenced in her CV
  missing_requirements   - comma-separated, or "None identified"
  eligibility_concerns   - what would stop her applying, or "None stated"
  worth_applying         - one of: YES, MAYBE, NO
  recommendation         - one concise sentence of advice

Internships:
{json.dumps(jobs, ensure_ascii=False)}
"""
        last_error = None
        for attempt in range(MAX_RETRIES):
            try:
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {"role": "system", "content": "Return structured JSON only."},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                )
                return parse_ai_review_json(response.choices[0].message.content)
            except Exception as error:
                last_error = error
                if attempt < MAX_RETRIES - 1:
                    time.sleep(2)
        print(f"  OpenRouter batch failed: {str(last_error)[:160]}")
        return []


if not RUN_AI_REVIEW:
    print("AI review is OFF.")
    print("Set RUN_AI_REVIEW = True, add OPENROUTER_API_KEY in Colab Secrets, then run this cell.")
    print("The deterministic workbook from the previous cell is already complete without it.")
else:
    api_key = get_openrouter_api_key()
    if not api_key:
        print("Missing OPENROUTER_API_KEY. Add it under Colab Secrets and re-run this cell.")
    else:
        provider = AIProvider(api_key)
        review_rows = sorted(active, key=lambda r: -r.get("Match Score", 0))
        if AI_REVIEW_LIMIT is not None:
            review_rows = review_rows[:AI_REVIEW_LIMIT]

        lookup, compact = {}, []
        for row_id, row in enumerate(review_rows, 1):
            lookup[row_id] = row
            compact.append(compact_job_for_ai(row, row_id))

        print(f"Reviewing {len(compact)} internships in batches of {AI_BATCH_SIZE} using {MODEL_NAME}...")
        reviewed = 0
        for start in range(0, len(compact), AI_BATCH_SIZE):
            batch = compact[start:start + AI_BATCH_SIZE]
            for review in provider.review(batch, CANDIDATE_PROFILE):
                row = lookup.get(int(review.get("row_id", 0) or 0))
                if not row:
                    continue
                row["AI Assessment"] = " ".join(filter(None, [
                    str(review.get("what_the_role_involves", "")).strip(),
                    str(review.get("why_it_matches", "")).strip(),
                    str(review.get("recommendation", "")).strip(),
                ]))
                row["AI Key Matching Skills"] = str(review.get("key_matching_skills", ""))
                row["AI Missing Requirements"] = str(review.get("missing_requirements", ""))
                row["AI Eligibility Concerns"] = str(review.get("eligibility_concerns", ""))
                row["AI Worth Applying"] = str(review.get("worth_applying", ""))
                reviewed += 1
            time.sleep(1)

        ai_top = [r for r in active
                  if r["Match Score"] >= TOP_MATCH_MIN_SCORE and not r["_eligibility_blocking"]]
        ai_deadlines = sorted([r for r in active if r["Deadline"]], key=lambda r: r.get("Days Left", 9999))
        with pd.ExcelWriter(AI_OUTPUT_FILE, engine="openpyxl") as writer:
            to_frame(active).to_excel(writer, index=False, sheet_name="All Internships")
            to_frame(ai_top).to_excel(writer, index=False, sheet_name="Top Matches")
            to_frame(ai_deadlines, [
                "Deadline", "Days Left", "Deadline Status", "Job Title", "Company",
                "Internship Type", "Eligibility", "Match Score", "AI Worth Applying", "Apply Link",
            ]).to_excel(writer, index=False, sheet_name="Deadline Tracker")
            df_tracker.to_excel(writer, index=False, sheet_name="Applied Tracker")

        ai_links = format_workbook(AI_OUTPUT_FILE)
        print(f"AI reviewed {reviewed} internships. Clickable links: {ai_links}")
        files.download(AI_OUTPUT_FILE)
        print(f"Downloaded {AI_OUTPUT_FILE}")
